# The Sentiment Agent (The "Newsroom")
This notebook implements an SVM paired with TfidfVectorizer to process financial news and determine if the prevailing mood is *Bullish* or *Bearish*.

It converts raw language into a sentiment probability score.

In [36]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [37]:
df = pd.read_csv('../csv-history/perfectly_balanced_stock_news_labeled.csv')
df.head()


,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,Article_Quality
0,12025.0,2023-12-16 22:00:00 UTC,My 6 Largest Portfolio Holdings Heading Into 2...,AAPL,https://www.nasdaq.com/articles/my-6-largest-p...,NaN,NaN,"After an absolute disaster of a year in 2022, ...",3: Apple There's little question that Apple (N...,3: Apple There's little question that Apple (N...,3: Apple There's little question that Apple (N...,3: Apple There's little question that Apple (N...,Rubbish
1,12026.0,2023-12-16 22:00:00 UTC,Brokers Suggest Investing in Apple (AAPL): Rea...,AAPL,https://www.nasdaq.com/articles/brokers-sugges...,NaN,NaN,"When deciding whether to buy, sell, or hold a ...",Let's take a look at what these Wall Street he...,Click to get this free report Apple Inc. (AAPL...,Let's take a look at what these Wall Street he...,Brokerage Recommendation Trends for AAPL Let's...,Rubbish
2,12027.0,2023-12-16 21:00:00 UTC,"Company News for Dec 19, 2023",AAPL,https://www.nasdaq.com/articles/company-news-f...,NaN,NaN,Shares of Apple Inc. AAPL lost 0.9% on China’s...,Shares of Apple Inc. AAPL lost 0.9% on China’s...,Click to get this free report Apple Inc. (AAPL...,Click to get this free report Apple Inc. (AAPL...,Click to get this free report Apple Inc. (AAPL...,Rubbish
3,12028.0,2023-12-16 21:00:00 UTC,NVIDIA (NVDA) Up 243% YTD: Will It Carry Momen...,AAPL,https://www.nasdaq.com/articles/nvidia-nvda-up...,NaN,NaN,NVIDIA Corporation NVDA has witnessed a remark...,Other Stocks in the $1T Club Apart from NVIDIA...,Other Stocks in the $1T Club Apart from NVIDIA...,Other Stocks in the $1T Club Apart from NVIDIA...,Other Stocks in the $1T Club Apart from NVIDIA...,Rubbish
4,12029.0,2023-12-16 21:00:00 UTC,"Pre-Market Most Active for Dec 19, 2023 : BMY,...",AAPL,https://www.nasdaq.com/articles/pre-market-mos...,NaN,NaN,The NASDAQ 100 Pre-Market Indicator is up 10.1...,"Apple Inc. (AAPL) is +0.86 at $196.75, with 1,...","Apple Inc. (AAPL) is +0.86 at $196.75, with 1,...","Apple Inc. (AAPL) is +0.86 at $196.75, with 1,...","Apple Inc. (AAPL) is +0.86 at $196.75, with 1,...",Rubbish


In [38]:
df = df.dropna(subset=['Article'])

In [39]:
df['Full_Text'] = df['Article_title'] + " " + df['Article']
X = df['Full_Text']
y = df['Article_Quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [41]:
svm_model = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)

print("Training the SVM model... This might take some time.")
svm_model.fit(X_train_tfidf, y_train)
print("Training complete!")

Training the SVM model... This might take some time.
Training complete!


In [42]:
y_pred = svm_model.predict(X_test_tfidf)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy Score: 0.5963455149501661

Classification Report:
               precision    recall  f1-score   support

        Good       0.66      0.69      0.67       366
     Rubbish       0.48      0.45      0.47       236

    accuracy                           0.60       602
   macro avg       0.57      0.57      0.57       602
weighted avg       0.59      0.60      0.59       602

